# ChibiCreate — BENCHMARK COMPARATIVO: WAI-illustrious-SDXL

> ## BENCHMARK COMPARATIVO — NAO E PIPELINE OFICIAL
>
> Responde **uma** pergunta: *o WAI-illustrious-SDXL preserva a roupa e o
> design original melhor que o FLUX.2 klein 4B?*
>
> Nao altera o benchmark FLUX, as Runs 001/002/003 do FLUX, o Flow 01, os
> quality gates nem o design transfer.

---

## Multi-referencia via IP-Adapter

O checkpoint SDXL nao tem mecanismo proprio de referencia como o
`ReferenceLatent` do FLUX. Isso **nao** quer dizer que ele nao consiga usar
varias referencias: o **IP-Adapter** fornece multi-referencia real.

```
full_body ──► IPAdapterEncoder (peso 1.0) ──┐
face      ──► IPAdapterEncoder (peso 0.6) ──┼──► IPAdapterCombineEmbeds
outfit    ──► IPAdapterEncoder (peso 0.8) ──┘             │
                                                  IPAdapterEmbeds
                                                          │
                                                      KSampler
```

Encoder + Combine em vez de `IPAdapterAdvanced` empilhado em serie, porque
so assim o peso de **cada** referencia fica explicito e auditavel.

| run | referencias | workflow |
|---|---|---|
| 001 | `full_body` | `v1` |
| 002 | `full_body` — repeticao exata da 001 | `v1` |
| 003 | `full_body` + `face` + `outfit` | `v2` |

### Sobre a comparacao com o FLUX

- **FLUX** = multi-referencia pelo mecanismo proprio (`ReferenceLatent`)
- **WAI** = multi-referencia por **IP-Adapter**

Sao implementacoes **diferentes**. A comparacao e sobre o **resultado visual
com as mesmas referencias de entrada**, nunca sobre equivalencia de
arquitetura.

### Dependencia de terceiro

O IP-Adapter exige o custom node `cubiq/ComfyUI_IPAdapter_plus` e dois pesos
auxiliares. A celula 6 pede **aceite explicito** antes de instalar. Se os
nodes nao aparecerem no servidor, o benchmark **PARA** — a Run 003 nunca cai
para uma referencia em silencio.


In [ ]:
#@title 0. PAINEL DO EXPERIMENTO — edite so esta celula { display-mode: "form" }
#@markdown # WAI CHIBI EXPERIMENT LAB
#@markdown Toda configuracao vive aqui. As demais celulas nao devem ser editadas.
#@markdown
#@markdown **Caminho estrutural (fixo, nao muda entre experimentos):**
#@markdown `full_body.png -> LoadImage -> VAEEncode -> latent -> KSampler -> VAEDecode`
#@markdown
#@markdown O IP-Adapter atua **adicionalmente sobre o MODEL**, nunca substituindo
#@markdown o latente. Nao existe `EmptyLatentImage` neste lab.

#@markdown ---
#@markdown ### PERSONAGEM E ENTRADAS
CHARACTER_ID = "waifu_001"  #@param {type:"string"}
SOURCE_IMAGE = "full_body.png"  #@param {type:"string"}
REFERENCE_FACE = "face.png"  #@param {type:"string"}
REFERENCE_OUTFIT = "outfit.png"  #@param {type:"string"}

#@markdown ---
#@markdown ### MODO DE REFERENCIA
#@markdown `BOTH` executa 1 REF e 3 REFS para a MESMA configuracao, permitindo
#@markdown comparar o efeito das referencias extras isoladamente.
REFERENCE_MODE = "BOTH"  #@param ["BOTH", "1_REF_FULL_BODY", "3_REF_FULL_BODY_FACE_OUTFIT"]

#@markdown ---
#@markdown ### PROMPT
#@markdown Presets genericos: descrevem o alvo estetico, nunca a personagem.
#@markdown Identidade e design vem das IMAGENS.
PROMPT_PRESET = "chibi_v1"  #@param ["chibi_v1"]
#@markdown
#@markdown **Ajuste manual.** Os campos ja vem com o texto do preset, entao
#@markdown da para editar a partir dele em vez de escrever do zero.
#@markdown Marque o toggle do lado para o texto editado valer; desmarcado,
#@markdown o preset e usado e o campo fica so como rascunho.
#@markdown
#@markdown O positivo tem de continuar GENERICO: descreve estilo, proporcao e
#@markdown qualidade, nunca a personagem. Cor de cabelo/olhos, chifres, roupa,
#@markdown acessorios ou nome sao BLOQUEADOS na celula 8 — essa informacao
#@markdown vem da imagem e das referencias, e coloca-la no texto tornaria a
#@markdown recipe inutil para as outras personagens.
USAR_PROMPT_CUSTOM = False  #@param {type:"boolean"}
PROMPT_CUSTOM = "1girl, solo, full body, standing, chibi, super deformed, large head, small body, short limbs, cute stylized anime character, anime coloring, clean lineart, simple cel shading, gacha game character, masterpiece, best quality, amazing quality"  #@param {type:"string"}
#@markdown
USAR_NEGATIVE_CUSTOM = False  #@param {type:"boolean"}
NEGATIVE_CUSTOM = "bad quality, worst quality, worst detail, sketch, watermark, signature, logo, text, multiple views, grid, multiple characters, realistic, photorealistic, 3d, semi-realistic, adult proportions, long limbs"  #@param {type:"string"}

#@markdown ---
#@markdown ### DENOISE
#@markdown Quanto da estrutura original pode ser transformada.
#@markdown Precisa ser < 1.0: com 1.0 o latente inicial e destruido.
DENOISE = 0.90  #@param {type:"slider", min:0.10, max:0.95, step:0.05}
#@markdown Sweep opcional. Deixe DESMARCADO para usar so o DENOISE acima.
USAR_SWEEP_DE_DENOISE = False  #@param {type:"boolean"}
DENOISE_SWEEP = "0.50, 0.60, 0.70, 0.80, 0.90"  #@param {type:"string"}

#@markdown ---
#@markdown ### SAMPLING
STEPS = 28  #@param {type:"slider", min:10, max:60, step:1}
CFG_SCALE = 5.5  #@param {type:"slider", min:1.0, max:12.0, step:0.5}
SAMPLER = "euler_ancestral"  #@param {type:"string"}
SCHEDULER = "normal"  #@param {type:"string"}
SEED = 42  #@param {type:"integer"}

#@markdown ---
#@markdown ### IP-ADAPTER (global)
IPADAPTER_WEIGHT = 0.75  #@param {type:"slider", min:0.0, max:1.5, step:0.05}
WEIGHT_TYPE = "linear"  #@param {type:"string"}
START_AT = 0.0  #@param {type:"slider", min:0.0, max:1.0, step:0.05}
END_AT = 1.0  #@param {type:"slider", min:0.0, max:1.0, step:0.05}
EMBEDS_SCALING = "V only"  #@param {type:"string"}

#@markdown ---
#@markdown ### PESOS POR REFERENCIA (modo 3 REFS)
FULL_BODY_WEIGHT = 1.0  #@param {type:"slider", min:0.0, max:2.0, step:0.05}
FACE_WEIGHT = 0.6  #@param {type:"slider", min:0.0, max:2.0, step:0.05}
OUTFIT_WEIGHT = 0.8  #@param {type:"slider", min:0.0, max:2.0, step:0.05}
COMBINE_METHOD = "concat"  #@param ["concat", "add", "subtract", "average", "norm average", "max", "min"]

#@markdown ---
#@markdown ### ROTULO DO EXPERIMENTO
#@markdown Vazio = rotulo automatico a partir dos parametros.
EXPERIMENT_LABEL = ""  #@param {type:"string"}

# ----------------------------------------------------------------------
import json, pathlib, datetime

MODEL_KEY = "wai_illustrious_sdxl_v170"
WORKFLOW = "experimental/wai_illustrious_ipadapter"

# Presets de prompt. GENERICOS: sem cabelo, olhos, chifres, roupa, cores
# ou nome de personagem. Essa informacao vem da imagem de entrada e das
# referencias — repeti-la em texto tornaria a recipe inutil para as
# outras personagens.
PROMPT_PRESETS = {
    "chibi_v1": {
        "positive": (
            "1girl, solo, full body, standing, chibi, super deformed, "
            "large head, small body, short limbs, cute stylized anime "
            "character, anime coloring, clean lineart, simple cel shading, "
            "gacha game character, masterpiece, best quality, amazing quality"
        ),
        "negative": (
            "bad quality, worst quality, worst detail, sketch, watermark, "
            "signature, logo, text, multiple views, grid, multiple "
            "characters, realistic, photorealistic, 3d, semi-realistic, "
            "adult proportions, long limbs"
        ),
    },
}

# O campo do form ja vem preenchido com o texto do preset, entao "campo nao
# vazio" nao significa mais "o usuario quis customizar" — quem decide e o
# toggle. Cada lado e independente.
_preset = PROMPT_PRESETS[PROMPT_PRESET]

def _escolhe(usar_custom, texto_custom, padrao, lado):
    if not usar_custom:
        return padrao, f"preset:{PROMPT_PRESET}"
    texto = texto_custom.strip()
    assert texto, (
        f"USAR_{lado}_CUSTOM esta marcado mas o campo esta vazio. "
        f"Desmarque para usar o preset, ou escreva o texto.")
    # Editar o campo e esquecer de mudar o texto nao e customizacao: manter
    # 'manual_override' na recipe faria dois experimentos identicos parecerem
    # diferentes.
    if texto == padrao.strip():
        return padrao, f"preset:{PROMPT_PRESET}"
    return texto, "manual_override"

PROMPT, _src_pos = _escolhe(
    USAR_PROMPT_CUSTOM, PROMPT_CUSTOM, _preset["positive"], "PROMPT")
NEGATIVE, _src_neg = _escolhe(
    USAR_NEGATIVE_CUSTOM, NEGATIVE_CUSTOM, _preset["negative"], "NEGATIVE")

# Registrado na recipe: sem isso, um experimento com prompt editado ficaria
# indistinguivel de um que usou o preset.
PROMPT_SOURCE = {"positive": _src_pos, "negative": _src_neg}
PROMPT_EDITADO = "manual_override" in PROMPT_SOURCE.values()

MODOS = {
    "1_REF_FULL_BODY": {
        "slug": "1_ref", "workflow_version": "v3", "reference_count": 1,
        "refs": ["full_body"],
    },
    "3_REF_FULL_BODY_FACE_OUTFIT": {
        "slug": "3_ref", "workflow_version": "v2", "reference_count": 3,
        "refs": ["full_body", "face", "outfit"],
    },
}
MODOS_A_EXECUTAR = (list(MODOS) if REFERENCE_MODE == "BOTH"
                    else [REFERENCE_MODE])

DENOISE_VALUES = (
    [round(float(x), 3) for x in DENOISE_SWEEP.split(",") if x.strip()]
    if USAR_SWEEP_DE_DENOISE else [float(DENOISE)])

for _d in DENOISE_VALUES:
    assert 0.0 < _d < 1.0, (
        f"denoise {_d} invalido: precisa ser < 1.0. Com 1.0 o latente "
        "inicial de full_body e destruido e o img2img vira txt2img.")
assert 0.0 <= START_AT < END_AT <= 1.0, "exige 0 <= START_AT < END_AT <= 1"

ARQUIVOS_REF = {"full_body": SOURCE_IMAGE, "face": REFERENCE_FACE,
                "outfit": REFERENCE_OUTFIT}

EXPERIMENT_STAMP = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
_auto = (f"d{DENOISE:.2f}_ipa{IPADAPTER_WEIGHT:.2f}_{WEIGHT_TYPE.replace(' ', '')}"
         f"_s{STEPS}_cfg{CFG_SCALE}")
EXPERIMENT_NAME = EXPERIMENT_LABEL.strip() or _auto
EXPERIMENT_DIR_NAME = f"experiment_{EXPERIMENT_STAMP}"

CONFIG = {
    "experiment": EXPERIMENT_NAME,
    "experiment_dir": EXPERIMENT_DIR_NAME,
    "created_utc": datetime.datetime.now(
        datetime.timezone.utc).isoformat(),
    "character_id": CHARACTER_ID,
    "model_key": MODEL_KEY,
    "pipeline": "img2img",
    "primary_image": "full_body",
    "primary_image_role": "source_image",
    "reference_mode": REFERENCE_MODE,
    "modes_to_run": MODOS_A_EXECUTAR,
    "reference_files": ARQUIVOS_REF,
    "prompt_preset": PROMPT_PRESET,
    "prompt": PROMPT,
    "negative_prompt": NEGATIVE,
    "prompt_source": PROMPT_SOURCE,
    "prompt_manually_edited": PROMPT_EDITADO,
    "prompt_type": "generic_chibi_base",
    "character_specific_prompt": False,
    "sampling": {
        "seed": int(SEED), "steps": int(STEPS), "cfg": float(CFG_SCALE),
        "sampler": SAMPLER, "scheduler": SCHEDULER,
    },
    "denoise_values": DENOISE_VALUES,
    "denoise_sweep_enabled": bool(USAR_SWEEP_DE_DENOISE),
    "ipadapter": {
        "weight": float(IPADAPTER_WEIGHT), "weight_type": WEIGHT_TYPE,
        "start_at": float(START_AT), "end_at": float(END_AT),
        "embeds_scaling": EMBEDS_SCALING,
    },
    "reference_weights": {
        "full_body": float(FULL_BODY_WEIGHT), "face": float(FACE_WEIGHT),
        "outfit": float(OUTFIT_WEIGHT),
    },
    "combine_method": COMBINE_METHOD,
    "status": "BASELINE_EXPERIMENTAL",
}

_total = len(MODOS_A_EXECUTAR) * len(DENOISE_VALUES)
print("=" * 64)
print("WAI CHIBI EXPERIMENT LAB")
print("=" * 64)
print("experimento :", EXPERIMENT_NAME)
print("diretorio   :", EXPERIMENT_DIR_NAME)
print("personagem  :", CHARACTER_ID)
print()
print("CAMINHO ESTRUTURAL (fixo)")
print(f"  {SOURCE_IMAGE} -> LoadImage -> VAEEncode -> latent -> KSampler")
print("  IP-Adapter atua sobre o MODEL, nao sobre o latente.")
print("  Sem EmptyLatentImage: nao e txt2img.")
print()
print("EXECUCOES PROGRAMADAS:", _total)
for _m in MODOS_A_EXECUTAR:
    _i = MODOS[_m]
    for _d in DENOISE_VALUES:
        print(f"  - {_i['slug']:6} denoise {_d:<5} "
              f"({_i['reference_count']} ref: {', '.join(_i['refs'])})")
print()
print("SAMPLING   : steps", STEPS, "| cfg", CFG_SCALE, "|", SAMPLER, "/", SCHEDULER,
      "| seed", SEED)
print("IP-ADAPTER : weight", IPADAPTER_WEIGHT, "|", WEIGHT_TYPE,
      f"| start {START_AT} end {END_AT}")
print("PESOS 3REF : full_body", FULL_BODY_WEIGHT, "| face", FACE_WEIGHT,
      "| outfit", OUTFIT_WEIGHT, "|", COMBINE_METHOD)
print()
print("full_body.png tem DOIS PAPEIS nos dois modos:")
print("  1. imagem inicial do img2img (VAEEncode)")
print("  2. referencia do IP-Adapter (IPAdapterEncoder)")
print()
print("PROMPT   :", PROMPT[:100] + ("..." if len(PROMPT) > 100 else ""))
print("  origem :", PROMPT_SOURCE["positive"])
print("NEGATIVO :", NEGATIVE[:100] + ("..." if len(NEGATIVE) > 100 else ""))
print("  origem :", PROMPT_SOURCE["negative"])
if PROMPT_EDITADO:
    print("  [prompt editado a mao — sera validado na celula 8]")
print("prompt generico: identidade vem das imagens, nao do texto.")
print()
print("[nota] SAMPLER, SCHEDULER e WEIGHT_TYPE sao validados contra o")
print("       /object_info do servidor na celula 8. Valor inexistente")
print("       bloqueia a execucao em vez de falhar na submissao.")


---

## Celula 1 — ambiente e repositorio

In [ ]:
#@title 1. Ambiente e repositorio { display-mode: "form" }
REPO_URL = "https://github.com/BloomRX/ChibiCreate"  #@param {type:"string"}
BRANCH   = "arena/01a07ece-chibicreate"  #@param {type:"string"}
ATUALIZAR_REPO = True  #@param {type:"boolean"}

import os, sys, subprocess, pathlib

%cd /content
if not pathlib.Path("/content/ChibiCreate/.git").exists():
    !git clone --branch $BRANCH $REPO_URL ChibiCreate
elif ATUALIZAR_REPO:
    !cd /content/ChibiCreate && git fetch origin $BRANCH && git checkout -B $BRANCH origin/$BRANCH

%cd /content/ChibiCreate
!git log --oneline -1

for _m in [k for k in list(sys.modules)
           if k.startswith("chibi") or k.startswith("scripts.chibi")]:
    del sys.modules[_m]
sys.path.insert(0, "/content/ChibiCreate")
sys.path.insert(0, "/content/ChibiCreate/scripts")

!pip install -q pyyaml pillow numpy


---

## Celula 2 — preflight (portao)

Se nao couber, **BLOCKED**. Sem fallback silencioso, sem CPU, sem trocar de
modelo, sem quantizar por conta propria.

In [ ]:
#@title 2. Preflight — GPU, VRAM, RAM, disco { display-mode: "form" }
import sys, json, shutil, subprocess, time

# Checkpoint SDXL fp16 (~8-10 GB) + IP-Adapter (~1 GB) + CLIP-Vision ViT-H
# (~2.5 GB no encode). Estimativa de registry, nao medicao nossa.
MIN_VRAM_GB = 12.0
MIN_DISK_GB = 20.0
MIN_RAM_GB  = 10.0

try:
    import torch
except ImportError:
    torch = None

print("=" * 62)
print("PREFLIGHT — detectado, nao presumido")
print("=" * 62)
print("Python :", sys.version.split()[0])
print("Torch  :", torch.__version__ if torch else "ausente")

if torch is None or not torch.cuda.is_available():
    print("CUDA   : INDISPONIVEL")
    print()
    print("=" * 62)
    print("BLOCKED — a sessao esta em CPU")
    print("=" * 62)
    print("Este notebook ja pede T4 por padrao (metadata accelerator=GPU,")
    print("gpuType=T4). O Colab, porem, ignora essa preferencia quando:")
    print("  - a copia aberta e antiga, salva antes desta correcao;")
    print("  - nao ha T4 disponivel na sua conta no momento;")
    print("  - a sessao foi reconectada como CPU apos expirar.")
    print()
    print("Corrija na sessao atual:")
    print("  Ambiente de execucao -> Alterar o tipo de ambiente de execucao")
    print("  -> Acelerador de hardware: GPU T4 -> Salvar")
    print()
    print("Se voce salvou uma copia no seu Drive, salve-a de novo DEPOIS de")
    print("trocar para T4: a preferencia fica gravada no arquivo .ipynb.")
    print()
    print("Nao ha fallback para CPU: SDXL em CPU levaria horas por imagem.")
    raise SystemExit("BLOCKED — sessao em CPU. Troque para GPU T4.")

props = torch.cuda.get_device_properties(0)
vram = props.total_memory / 1024 ** 3
free_disk = shutil.disk_usage("/content").free / 1024 ** 3
try:
    import psutil
    ram = psutil.virtual_memory().total / 1024 ** 3
except ImportError:
    ram = float(subprocess.check_output(
        ["awk", "/MemTotal/ {print $2/1048576}", "/proc/meminfo"]).strip())

GPU_INFO = {
    "name": props.name, "vram_total_gb": round(vram, 2),
    "vram_free_gb": round(torch.cuda.mem_get_info()[0] / 1024 ** 3, 2),
    "cuda": torch.version.cuda,
    "capability": f"{props.major}.{props.minor}",
    "torch": torch.__version__, "python": sys.version.split()[0],
    "bf16_supported": props.major >= 8,
    "ram_gb": round(ram, 2), "disk_free_gb": round(free_disk, 2),
}
for k, v in GPU_INFO.items():
    print(f"  {k:18} {v}")

print()
falhas = []
if vram < MIN_VRAM_GB:
    falhas.append(f"VRAM {vram:.1f} < {MIN_VRAM_GB} GB")
if free_disk < MIN_DISK_GB:
    falhas.append(f"disco {free_disk:.1f} < {MIN_DISK_GB} GB")
if ram < MIN_RAM_GB:
    falhas.append(f"RAM {ram:.1f} < {MIN_RAM_GB} GB")

if not GPU_INFO["bf16_supported"]:
    print("AVISO: sem bf16 nativo (capability < 8.0, ex. T4). Cai para fp16.")

if falhas:
    print("=" * 62); print("BLOCKED"); print("=" * 62)
    for f in falhas:
        print(" -", f)
    print()
    print("Nao fazer fallback silencioso: nao trocar de modelo, nao remover")
    print("referencias, nao quantizar por conta propria. Reportar o bloqueio.")
    raise SystemExit("BLOCKED")

print("Hardware adequado.")
json.dump(GPU_INFO, open("/content/gpu_info_wai.json", "w"), indent=2)


---

## Celula 3 — entradas e hashes

As **mesmas** referencias do benchmark FLUX, derivadas de
`characters/<CHARACTER_ID>/reference/`. Os originais nunca sao modificados.

In [ ]:
#@title 3. Referencias — full_body, face, outfit { display-mode: "form" }
import hashlib, pathlib, json
import numpy as np
from PIL import Image

try:
    CONFIG, MODOS_A_EXECUTAR, CHARACTER_ID, ARQUIVOS_REF
except NameError:
    raise SystemExit("BLOCKED — execute a celula 0 antes desta.")

# Carregamos SEMPRE as tres: o modo BOTH precisa das tres, e ter as
# tres validadas de antemao evita descobrir um arquivo faltando no meio
# do lote, depois de ja ter gastado uma execucao.
REFS_NECESSARIAS = sorted({r for m in MODOS_A_EXECUTAR
                           for r in MODOS[m]["refs"]})

REF_DIR = pathlib.Path(
    f"/content/ChibiCreate/characters/{CHARACTER_ID}/reference")
ARQUIVOS = dict(ARQUIVOS_REF)

# Papel de cada referencia nesta run. full_body e SEMPRE a imagem de
# partida do img2img; na Run 003 ela acumula o papel de referencia do
# IP-Adapter.
def _papeis(papel):
    if papel not in REFS_NECESSARIAS:
        return []
    if papel == "full_body":
        # nos DOIS modos full_body e latente inicial E referencia
        return ["source_image", "ipadapter_reference"]
    return ["ipadapter_reference"]

ENTRADAS = {}
for papel, arq in ARQUIVOS.items():
    p = REF_DIR / arq
    if not p.exists():
        print(f"  [ausente] {papel}: {p}")
        continue
    b = p.read_bytes()
    im = Image.open(p)
    ENTRADAS[papel] = {
        "file": arq, "path": str(p),
        "artifact_sha256": hashlib.sha256(b).hexdigest(),
        "pixel_sha256": hashlib.sha256(
            np.array(im.convert("RGBA")).tobytes()).hexdigest(),
        "size": list(im.size), "bytes": len(b),
        "usada_neste_experimento": papel in REFS_NECESSARIAS,
        "roles": _papeis(papel),
    }
    marca = ("+".join(_papeis(papel)) if papel in REFS_NECESSARIAS
             else "nao usada neste experimento")
    print(f"  {papel:10} {str(im.size):12} "
          f"{ENTRADAS[papel]['artifact_sha256'][:16]}  [{marca}]")

faltando = [r for r in REFS_NECESSARIAS if r not in ENTRADAS]
if faltando:
    raise SystemExit(
        f"BLOCKED — este experimento exige {REFS_NECESSARIAS} e faltam "
        f"{faltando}. Nao executar com menos referencias em silencio.")

# A imagem de partida define a resolucao de saida: nao ha resize.
if "full_body" not in ENTRADAS:
    raise SystemExit(
        "BLOCKED — full_body.png e a imagem inicial do img2img em TODAS as "
        "runs. Sem ela nao ha o que converter em chibi.")
SRC_W, SRC_H = ENTRADAS["full_body"]["size"]

print()
print(f"{len(REFS_NECESSARIAS)} referencia(s) confirmada(s):",
      ", ".join(REFS_NECESSARIAS))
print(f"Imagem de partida : full_body.png  {SRC_W}x{SRC_H}")
print(f"Resolucao de saida: {SRC_W}x{SRC_H} (herdada, sem resize — "
      "redimensionar deformaria a arte)")
print("full_body em papel DUPLO: latente inicial + referencia IP-Adapter.")
print("Arquivos NAO sao modificados — apenas lidos.")
json.dump(ENTRADAS, open("/content/wai_inputs.json", "w"), indent=2)


---

## Celula 4 — Google Drive: localizar e validar o checkpoint

O checkpoint ja existe no seu Drive. Esta celula **monta**, **localiza**,
**valida** e **liga** o arquivo ao ComfyUI — sem download, sem upload e sem
tocar no Civitai.

O arquivo original **nao e movido nem modificado**: tentamos symlink
primeiro (instantaneo) e so caimos para copia se o symlink nao funcionar
neste ambiente.

Sobre a versao: o arquivo e explicitamente `waiIllustriousSDXL_v170`. O
`modelVersionId` do Civitai fica `unknown/pending` se voce nao souber — isso
**nao** impede a execucao, porque a identidade real do arquivo e garantida
pelo **SHA256**, que e mais forte que um id de catalogo.


In [ ]:
#@title 4. Montar o Drive e validar o checkpoint { display-mode: "form" }
#@markdown Caminho DENTRO do seu Drive (sem `/content/drive/MyDrive/`).
CKPT_DRIVE_PATH = "ComfyUI_Data/models/checkpoints/waiIllustriousSDXL_v170.safetensors"  #@param {type:"string"}
#@markdown Metadados do Civitai. Opcionais: deixe vazio e ficam
#@markdown `unknown/pending` — o SHA256 e que identifica o arquivo.
CIVITAI_MODEL_ID = "827184"  #@param {type:"string"}
CIVITAI_VERSION_ID = ""  #@param {type:"string"}
LICENCA_EXIBIDA = "Commercial use allowed (conforme UI do Civitai)"  #@param {type:"string"}

import hashlib, json, os, pathlib, struct, time

from chibi import model_registry as mr

CFG = mr.get_model(MODEL_KEY)

# ---------------------------------------------------------------- 1. mount
DRIVE_ROOT = pathlib.Path("/content/drive")
if not (DRIVE_ROOT / "MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Drive ja montado.")

MYDRIVE = DRIVE_ROOT / "MyDrive"
if not MYDRIVE.exists():
    raise SystemExit("BLOCKED — Drive nao montado. Reexecute e autorize.")

# ------------------------------------------------------------- 2. localizar
origem = (MYDRIVE / CKPT_DRIVE_PATH).expanduser()
CAMINHO_LOGICO = f"My Drive/{CKPT_DRIVE_PATH}"

if not origem.exists():
    print("=" * 64)
    print("BLOCKED — checkpoint nao encontrado no caminho informado")
    print("=" * 64)
    print("procurado :", origem)
    print("logico    :", CAMINHO_LOGICO)
    print()
    pasta = origem.parent
    if pasta.exists():
        achados = sorted(p.name for p in pasta.glob("*.safetensors"))
        print(f"A pasta existe e contem {len(achados)} .safetensors:")
        for nome in achados[:25]:
            print("   -", nome)
        if not achados:
            print("   (nenhum)")
    else:
        print("A pasta nao existe:", pasta)
        anc = pasta
        while anc != MYDRIVE and not anc.exists():
            anc = anc.parent
        print("Ancestral existente:", anc)
        if anc.exists():
            subs = sorted(p.name for p in anc.iterdir() if p.is_dir())[:25]
            print("Subpastas:", subs or "(nenhuma)")
    print()
    print("Corrija CKPT_DRIVE_PATH no formulario acima e reexecute.")
    print("Nao vamos adivinhar nome de arquivo nem baixar do Civitai.")
    raise SystemExit("BLOCKED — informe o caminho correto do Drive.")

tamanho = origem.stat().st_size

# ----------------------------------------------- 3. validar antes de usar
def ler_header_safetensors(p):
    """Le o header JSON do safetensors sem carregar os pesos.

    Layout: 8 bytes little-endian com o tamanho do header, seguido do JSON.
    """
    with open(p, "rb") as f:
        (n,) = struct.unpack("<Q", f.read(8))
        if not (0 < n < 200_000_000):
            raise ValueError(f"tamanho de header implausivel: {n}")
        return json.loads(f.read(n).decode("utf-8"))

print()
print("Validando o arquivo (header, sem carregar os pesos)...")
try:
    header = ler_header_safetensors(origem)
except Exception as exc:
    raise SystemExit(
        f"BLOCKED — nao e um .safetensors valido: {type(exc).__name__}: {exc}")

chaves = [k for k in header if k != "__metadata__"]

# Assinaturas de arquitetura. O segundo text encoder (OpenCLIP bigG) e o
# que distingue SDXL de SD 1.5/2.x; o primeiro bloco do UNet confirma que
# ha um modelo de difusao aqui, e nao um LoRA ou um VAE solto.
tem_unet = any(k.startswith("model.diffusion_model.") for k in chaves)
tem_te1 = any(k.startswith("conditioner.embedders.0.") for k in chaves)
tem_te2 = any(k.startswith("conditioner.embedders.1.") for k in chaves)
tem_vae = any(k.startswith("first_stage_model.") for k in chaves)

print(f"  tensores               : {len(chaves)}")
print(f"  UNet                   : {'sim' if tem_unet else 'NAO'}")
print(f"  text encoder 1 (CLIP-L) : {'sim' if tem_te1 else 'NAO'}")
print(f"  text encoder 2 (bigG)   : {'sim' if tem_te2 else 'NAO'}  <- marca do SDXL")
print(f"  VAE                    : {'sim' if tem_vae else 'NAO'}")

problemas = []
if not tem_unet:
    problemas.append("sem UNet (model.diffusion_model.*) — nao e checkpoint")
if not tem_te2:
    problemas.append(
        "sem o 2o text encoder (conditioner.embedders.1.*) — nao e SDXL")
if tamanho < 3 * 1024 ** 3:
    problemas.append(
        f"tamanho {tamanho / 1024 ** 3:.2f} GB e pequeno demais para SDXL")

if problemas:
    print()
    print("=" * 64); print("BLOCKED — nao parece um checkpoint SDXL"); print("=" * 64)
    for x in problemas:
        print(" -", x)
    print()
    print("Nao vamos iniciar a inferencia com um arquivo invalido.")
    raise SystemExit("BLOCKED — checkpoint invalido")

meta_embutida = header.get("__metadata__", {}) or {}

# ----------------------------------------------------------- 4. SHA256
print()
print(f"Calculando SHA256 de {tamanho / 1024 ** 3:.2f} GB (leitura via Drive,")
print("pode levar alguns minutos)...")
t0 = time.time()
h = hashlib.sha256()
with open(origem, "rb") as f:
    for bloco in iter(lambda: f.read(1 << 23), b""):
        h.update(bloco)
CKPT_SHA256 = h.hexdigest()
print(f"  concluido em {time.time() - t0:.0f}s")

# --------------------------------------- 5. ligar ao ComfyUI (sem copiar)
CKPT_DIR = pathlib.Path("/content/ComfyUI/models/checkpoints")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
destino = CKPT_DIR / origem.name

metodo = None
if destino.is_symlink() or destino.exists():
    metodo = "ja_presente"
else:
    try:
        destino.symlink_to(origem)
        # Symlink so serve se der para LER de verdade atraves dele.
        with open(destino, "rb") as f:
            f.read(8)
        metodo = "symlink"
    except Exception as exc:
        print(f"  symlink indisponivel ({type(exc).__name__}); copiando...")
        if destino.is_symlink():
            destino.unlink()
        import shutil
        shutil.copy2(origem, destino)
        metodo = "copia"

# ------------------------------------------------------------- 6. relatorio
print()
print("=" * 64)
print("CHECKPOINT PRONTO")
print("=" * 64)
print(f"  caminho encontrado : {origem}")
print(f"  caminho logico     : {CAMINHO_LOGICO}")
print(f"  nome               : {origem.name}")
print(f"  tamanho            : {tamanho:,} bytes ({tamanho / 1024 ** 3:.2f} GB)")
print(f"  sha256             : {CKPT_SHA256}")
print(f"  origem             : google_drive")
print(f"  ligado ao ComfyUI  : {metodo} -> {destino}")
if meta_embutida:
    print(f"  metadata embutida  : {dict(list(meta_embutida.items())[:5])}")
print()
print("  Arquivo original do Drive NAO foi movido nem modificado.")
print("  Civitai NAO foi acessado.")

VERSAO = {
    "source": "google_drive",
    "drive_logical_path": CAMINHO_LOGICO,
    "drive_absolute_path": str(origem),
    "file": origem.name,
    "sha256": CKPT_SHA256,
    "size_bytes": tamanho,
    "link_method": metodo,
    "comfy_path": str(destino),
    "civitai_model_id": CIVITAI_MODEL_ID or "unknown/pending",
    "civitai_model_version_id": CIVITAI_VERSION_ID or "unknown/pending",
    "revision_from_filename": "v170",
    "license_displayed": LICENCA_EXIBIDA,
    "verified_by_agent": False,
    "sdxl_validated": True,
    "sdxl_validation": {
        "tensor_count": len(chaves), "has_unet": tem_unet,
        "has_text_encoder_1": tem_te1, "has_text_encoder_2": tem_te2,
        "has_vae": tem_vae,
    },
    "embedded_metadata": meta_embutida,
}

if VERSAO["civitai_model_version_id"] == "unknown/pending":
    print()
    print("[HUMAN REVIEW REQUIRED] modelVersionId: unknown/pending.")
    print("Nao bloqueia a execucao: o SHA256 acima identifica o arquivo de")
    print("forma mais forte que um id de catalogo. Voce pode preencher os")
    print("metadados depois — o hash ja esta gravado no recipe.")

json.dump(VERSAO, open("/content/wai_version.json", "w"), indent=2)


---

## Celula 5 — ComfyUI

In [ ]:
#@title 5. Subir o ComfyUI { display-mode: "form" }
import subprocess, time, urllib.request, json, pathlib, os, shutil

COMFY = pathlib.Path("/content/ComfyUI")
COMFY_REPO = "https://github.com/comfyanonymous/ComfyUI.git"

def comfy_no_ar(timeout=5):
    try:
        with urllib.request.urlopen(
                "http://127.0.0.1:8188/system_stats", timeout=timeout) as r:
            return json.load(r)
    except Exception:
        return None

# A celula 4 pode ter criado /content/ComfyUI/models/checkpoints para
# colocar o symlink do Drive. Entao "a pasta existe" NAO significa "o
# ComfyUI esta instalado" — o marcador real e o .git + o main.py.
instalado = (COMFY / ".git").is_dir() and (COMFY / "main.py").is_file()

if not instalado:
    if COMFY.exists() and any(COMFY.iterdir()):
        # git clone recusa diretorio nao vazio: clonamos ao lado e
        # mesclamos, preservando o que a celula 4 ja colocou.
        print("Pasta ja existe (checkpoint do Drive). Clonando e mesclando...")
        tmp = pathlib.Path("/content/_comfy_tmp")
        if tmp.exists():
            shutil.rmtree(tmp)
        subprocess.run(["git", "clone", "-q", COMFY_REPO, str(tmp)], check=True)
        for item in tmp.iterdir():
            destino = COMFY / item.name
            if not destino.exists():
                shutil.move(str(item), str(destino))
            elif item.is_dir():
                # models/ ja existe com o checkpoint: copia so o que falta.
                for sub in item.rglob("*"):
                    rel = sub.relative_to(item)
                    alvo = destino / rel
                    if sub.is_dir():
                        alvo.mkdir(parents=True, exist_ok=True)
                    elif not alvo.exists():
                        alvo.parent.mkdir(parents=True, exist_ok=True)
                        shutil.move(str(sub), str(alvo))
        shutil.rmtree(tmp, ignore_errors=True)
    else:
        subprocess.run(["git", "clone", "-q", COMFY_REPO, str(COMFY)], check=True)

    !pip install -q -r /content/ComfyUI/requirements.txt

if not (COMFY / ".git").is_dir():
    raise SystemExit(
        f"BLOCKED — {COMFY} nao e um clone do ComfyUI. Apague a pasta "
        "(preservando models/) e reexecute esta celula.")

COMFY_COMMIT = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=COMFY).decode().strip()
print("ComfyUI commit:", COMFY_COMMIT)

# O symlink do checkpoint tem de ter sobrevivido a mesclagem.
try:
    _ck = pathlib.Path(VERSAO["comfy_path"])
    print("checkpoint    :", _ck.name,
          "(ok)" if _ck.exists() else "AUSENTE — reexecute a celula 4")
except NameError:
    print("checkpoint    : celula 4 ainda nao executada")

# Marcador estavel para achar o processo depois. Sem isso o pkill teria
# de adivinhar a linha de comando: o processo sobe como "python main.py"
# (cwd=/content/ComfyUI), entao um pkill -f "ComfyUI/main.py" NAO casa.
COMFY_TAG = "chibi_wai_comfy"


def matar_comfy(verbose=True):
    """Derruba QUALQUER ComfyUI ouvindo na 8188 e so retorna quando cair.

    Necessario antes de carregar custom nodes novos: o ComfyUI le
    custom_nodes uma unica vez, no boot. Reaproveitar um servidor que
    subiu antes da instalacao faz os nodes novos simplesmente nao
    existirem — sem erro alguma, o que e o pior caso.
    """
    global PROC
    try:
        if PROC is not None:
            PROC.terminate()
            try:
                PROC.wait(timeout=30)
            except Exception:
                PROC.kill()
    except NameError:
        pass
    PROC = None

    # Mata por marcador, por padrao de comando e por porta — o processo
    # pode ter sobrevivido a um restart de kernel, sem Popen para nós.
    subprocess.run(["pkill", "-f", COMFY_TAG], check=False)
    subprocess.run(["pkill", "-f", "main.py --listen"], check=False)
    subprocess.run(["fuser", "-k", "8188/tcp"], check=False,
                   capture_output=True)

    for i in range(30):
        if comfy_no_ar(timeout=2) is None:
            if verbose:
                print(f"  servidor anterior derrubado (~{i}s)")
            return True
        time.sleep(2)
        if i == 14:
            subprocess.run(["pkill", "-9", "-f", "main.py --listen"],
                           check=False)
            subprocess.run(["fuser", "-k", "-9", "8188/tcp"], check=False,
                           capture_output=True)
    raise SystemExit(
        "BLOCKED — nao foi possivel derrubar o ComfyUI na porta 8188. "
        "Use Runtime > Restart session e execute de novo a partir da "
        "celula 4.")


def subir_comfy(force=False):
    """Sobe o ComfyUI. Com force=True, derruba o anterior antes.

    force e obrigatorio depois de instalar custom nodes.
    """
    global PROC
    if force:
        matar_comfy()
    elif comfy_no_ar():
        print("  ja estava no ar; reaproveitando.")
        PROC = None
        return None
    log = open("/content/comfyui_wai.log", "w")
    p = subprocess.Popen(
        ["python", "main.py", "--listen", "127.0.0.1", "--port", "8188"],
        cwd=str(COMFY), stdout=log, stderr=subprocess.STDOUT,
        env={**os.environ, "CHIBI_COMFY_TAG": COMFY_TAG})
    for i in range(120):
        time.sleep(5)
        if comfy_no_ar():
            print(f"  no ar apos ~{(i + 1) * 5}s")
            PROC = p
            return p
        if p.poll() is not None:
            print(open("/content/comfyui_wai.log").read()[-3000:])
            raise SystemExit("ComfyUI morreu ao iniciar")
    raise SystemExit("ComfyUI nao respondeu em 10 min")


PROC = subir_comfy()
os.environ["CHIBI_COMFY_URL"] = "http://127.0.0.1:8188"


---

## Celula 6 — IP-Adapter (**pule nas Runs 001/002**)

O baseline nao usa IP-Adapter. Esta celula e a proxima **so sao necessarias
para a Run 003** — nas Runs 001/002 elas se autodesativam e nao instalam
nada.

| item | arquivo | licenca |
|---|---|---|
| custom node | `cubiq/ComfyUI_IPAdapter_plus` | Apache-2.0 |
| adapter | `ip-adapter-plus_sdxl_vit-h.safetensors` | Apache-2.0 |
| encoder | `CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors` | Apache-2.0 |


In [ ]:
#@title 6. Instalar IP-Adapter (so na Run 003) { display-mode: "form" }
#@markdown O lab usa IP-Adapter nos DOIS modos (1 REF e 3 REFS), entao
#@markdown esta instalacao e sempre necessaria.
ACEITO_INSTALAR_IPADAPTER = False  #@param {type:"boolean"}

import pathlib, subprocess, hashlib, json, os

def sha256_of(caminho, chunk=1 << 20):
    """SHA256 lendo em blocos: os pesos nao cabem confortavelmente em RAM."""
    h = hashlib.sha256()
    with open(caminho, "rb") as fh:
        for bloco in iter(lambda: fh.read(chunk), b""):
            h.update(bloco)
    return h.hexdigest()

IPADAPTER_META = None

if False:  # o lab sempre usa IP-Adapter
    pass
else:
    IPA_DIR = pathlib.Path(
        "/content/ComfyUI/custom_nodes/ComfyUI_IPAdapter_plus")
    MODELS = pathlib.Path("/content/ComfyUI/models")
    CFG_IPA = CFG["ipadapter_models"]

    if not ACEITO_INSTALAR_IPADAPTER and not IPA_DIR.exists():
        print("=" * 62)
        print("BLOCKED — IP-Adapter nao instalado")
        print("=" * 62)
        print("custom node :", CFG["custom_node_repo"])
        print("nodes       :", ", ".join(CFG["custom_node_nodes"]))
        print()
        print("Sem ele nao ha multi-referencia. A Run 003 NAO roda com uma")
        print("referencia so — seria substituir multi-reference em silencio.")
        raise SystemExit("Marque ACEITO_INSTALAR_IPADAPTER para prosseguir.")

    if not IPA_DIR.exists():
        !git clone -q {CFG["custom_node_repo"]} {IPA_DIR}

    IPA_COMMIT = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=IPA_DIR).decode().strip()
    IPA_TAG = subprocess.run(["git", "describe", "--tags", "--always"],
                             cwd=IPA_DIR, capture_output=True,
                             text=True).stdout.strip()
    print("IP-Adapter node:", CFG["custom_node_repo"])
    print("  commit :", IPA_COMMIT)
    print("  tag    :", IPA_TAG)

    IPADAPTER_ASSETS = {}
    for papel, spec in CFG_IPA.items():
        destino = MODELS / spec["target_dir"].split("/", 1)[1] / spec["file"]
        destino.parent.mkdir(parents=True, exist_ok=True)
        if not destino.exists():
            url = (f"https://huggingface.co/{spec['repo']}/resolve/main/"
                   f"{spec['repo_path']}")
            print(f"  baixando {papel}: {spec['file']}")
            !wget -q --show-progress -O "{destino}" "{url}"
        if not destino.exists() or destino.stat().st_size < 1_000_000:
            raise SystemExit(
                f"BLOCKED — download de '{spec['file']}' falhou "
                f"({destino}). Nao seguir sem o peso.")
        real = sha256_of(destino)
        IPADAPTER_ASSETS[papel] = {
            "file": spec["file"], "repo": spec["repo"],
            "repo_path": spec["repo_path"], "license": spec["license"],
            "sha256": real, "size_bytes": destino.stat().st_size,
        }
        print(f"  {papel:11} {spec['file']}")
        print(f"    sha256   {real}")
        print(f"    tamanho  {destino.stat().st_size / 1e6:.0f} MB"
              f" | {spec['license']}")

    IPADAPTER_META = {
        "repo": CFG["custom_node_repo"],
        "commit": IPA_COMMIT, "revision": IPA_TAG,
        "nodes": CFG["custom_node_nodes"], "assets": IPADAPTER_ASSETS,
    }
    json.dump(IPADAPTER_META, open("/content/ipadapter_meta.json", "w"),
              indent=2)
    print()
    print("A proxima celula REINICIA o ComfyUI automaticamente: os\n"
          "custom nodes so sao lidos no boot do servidor.")


In [ ]:
#@title 7. Reiniciar o ComfyUI e carregar os custom nodes { display-mode: "form" }
import subprocess, time, urllib.request, json, pathlib, re

def _carregar_object_info():
    with urllib.request.urlopen(
            "http://127.0.0.1:8188/object_info", timeout=180) as r:
        return json.load(r)

if False:  # o lab sempre carrega custom nodes
    pass
else:
    # RESTART OBRIGATORIO. O ComfyUI le custom_nodes so no boot: se o
    # servidor subiu na celula 5, antes do git clone da celula 6, os nodes
    # do IP-Adapter nao existem nele. Reaproveitar o processo antigo faz
    # os nodes "sumirem" sem nenhum erro.
    print("Reiniciando o ComfyUI para carregar os nodes do IP-Adapter...")
    PROC = subir_comfy(force=True)

    OBJECT_INFO = _carregar_object_info()
    print("nodes no servidor:", len(OBJECT_INFO))
    print()

    faltando = [n for n in CFG["custom_node_nodes"] if n not in OBJECT_INFO]
    for n in CFG["custom_node_nodes"]:
        print(f"  {'OK  ' if n in OBJECT_INFO else 'FALTA'} {n}")

    if faltando:
        print()
        print("=" * 62)
        print("BLOCKED — nodes IP-Adapter ausentes apos o restart")
        print("=" * 62)
        print("faltando:", faltando)
        print()

        IPA_DIR = pathlib.Path(
            "/content/ComfyUI/custom_nodes/ComfyUI_IPAdapter_plus")
        print("custom node no disco :", IPA_DIR.exists())
        if IPA_DIR.exists():
            print("  __init__.py        :", (IPA_DIR / "__init__.py").exists())
            print("  arquivos           :",
                  len(list(IPA_DIR.glob('*.py'))), "arquivos .py")
        ipa_models = pathlib.Path("/content/ComfyUI/models/ipadapter")
        cv_models = pathlib.Path("/content/ComfyUI/models/clip_vision")
        for d in (ipa_models, cv_models):
            print(f"  {d.name:18}:",
                  [f.name for f in d.glob('*')] if d.exists() else "AUSENTE")

        # A causa real quase sempre esta no log, numa linha de import.
        log = pathlib.Path("/content/comfyui_wai.log")
        if log.exists():
            texto = log.read_text(errors="replace")
            linhas = texto.split("\n")
            relevantes = [
                l for l in linhas
                if re.search(r"ipadapter|import fail|cannot import|"
                             r"ModuleNotFoundError|Traceback|error", l, re.I)]
            if relevantes:
                print()
                print("--- log do ComfyUI (linhas relevantes) ---")
                for l in relevantes[-40:]:
                    print("  ", l[:200])
        print()
        print("Se o log mostrar erro de import, e incompatibilidade entre o")
        print("custom node e a versao do ComfyUI. NAO substituir")
        print("multi-reference por uma imagem so.")
        raise SystemExit("BLOCKED — IP-Adapter indisponivel")

    print()
    print("IP-Adapter disponivel. Multi-referencia possivel.")
    print()
    # Os nomes existirem nao basta: os campos usados no grafo tambem
    # precisam existir, senao o erro so aparece na submissao.
    _enc = OBJECT_INFO["IPAdapterEncoder"]["input"]
    _campos = {**_enc.get("required", {}), **_enc.get("optional", {})}
    print("IPAdapterEncoder aceita:", ", ".join(sorted(_campos)))
    for campo in ("ipadapter", "image", "weight", "clip_vision"):
        if campo not in _campos:
            print(f"  [atencao] campo '{campo}' nao existe neste node — a"
                  " assinatura mudou; o grafo pode falhar na submissao.")


---

## Celula 8 — validar o grafo

Confere o workflow da run escolhida contra os nodes reais do servidor. Se
falhar, **pare** — nao edite o grafo para "fazer passar".

In [ ]:
#@title 8. Validar nodes, parametros e grafos { display-mode: "form" }
import json, hashlib, pathlib

falhas = []
def checa(cond, ok, erro):
    print(("  OK    " if cond else "  FALHA ") + (ok if cond else erro))
    if not cond:
        falhas.append(erro)

print("=" * 64)
print("1. NODES DISPONIVEIS NO SERVIDOR")
print("=" * 64)
NODES_NECESSARIOS = {
    "1_REF_FULL_BODY": ["IPAdapterModelLoader", "CLIPVisionLoader",
                        "IPAdapterEncoder", "IPAdapterEmbeds"],
    "3_REF_FULL_BODY_FACE_OUTFIT": ["IPAdapterModelLoader", "CLIPVisionLoader",
                                    "IPAdapterEncoder",
                                    "IPAdapterCombineEmbeds",
                                    "IPAdapterEmbeds"],
}
_precisa = sorted({n for m in MODOS_A_EXECUTAR for n in NODES_NECESSARIOS[m]})
for n in _precisa:
    checa(n in OBJECT_INFO, f"{n} disponivel", f"{n} AUSENTE no servidor")

# O guia externo cita "Character Reference". Registramos o que EXISTE no
# servidor, sem afirmar equivalencia com FaceID ou PLUS FACE — isso e
# hipotese a testar, nao fato.
_relacionados = sorted(k for k in OBJECT_INFO
                       if "IPAdapter" in k or "FaceID" in k)
print()
print(f"  [info] {len(_relacionados)} nodes IPAdapter/FaceID no servidor:")
for k in _relacionados:
    print("        -", k)

if falhas:
    print()
    print("NAO degradar para menos referencias em silencio.")
    raise SystemExit(f"BLOCKED — nodes ausentes: {falhas}")

print()
print("=" * 64)
print("2. PARAMETROS REAIS DOS NODES (nada inventado)")
print("=" * 64)

def _opcoes(node, campo):
    """Lista de opcoes que o servidor aceita para um campo, ou None."""
    spec = OBJECT_INFO[node]["input"]
    for grupo in ("required", "optional"):
        if campo in spec.get(grupo, {}):
            val = spec[grupo][campo][0]
            return val if isinstance(val, list) else None
    return "AUSENTE"

SAMPLERS = _opcoes("KSampler", "sampler_name") or []
SCHEDULERS = _opcoes("KSampler", "scheduler") or []
checa(SAMPLER in SAMPLERS, f"sampler '{SAMPLER}' suportado",
      f"sampler '{SAMPLER}' nao existe. Disponiveis: {SAMPLERS}")
checa(SCHEDULER in SCHEDULERS, f"scheduler '{SCHEDULER}' suportado",
      f"scheduler '{SCHEDULER}' nao existe. Disponiveis: {SCHEDULERS}")

_emb = OBJECT_INFO["IPAdapterEmbeds"]["input"]
CAMPOS_EMBEDS = {**_emb.get("required", {}), **_emb.get("optional", {})}
print()
print("  IPAdapterEmbeds aceita:", ", ".join(sorted(CAMPOS_EMBEDS)))

# Parametros do painel que dependem do node. Se o node instalado nao
# suportar algum, avisamos em vez de fingir que o controle funciona.
PARAMS_SUPORTADOS = {}
for campo, valor in (("weight_type", WEIGHT_TYPE),
                     ("embeds_scaling", EMBEDS_SCALING)):
    opts = _opcoes("IPAdapterEmbeds", campo)
    PARAMS_SUPORTADOS[campo] = opts != "AUSENTE"
    if opts == "AUSENTE":
        print(f"  [atencao] '{campo}' NAO existe neste node: o controle da "
              "celula 0 sera ignorado.")
    elif isinstance(opts, list):
        checa(valor in opts, f"{campo} '{valor}' suportado",
              f"{campo} '{valor}' invalido. Disponiveis: {opts}")
    else:
        print(f"  OK    {campo} aceito (campo livre)")
for campo in ("start_at", "end_at", "weight"):
    existe = campo in CAMPOS_EMBEDS
    PARAMS_SUPORTADOS[campo] = existe
    checa(existe, f"{campo} suportado", f"{campo} nao existe no node")

_cmb = _opcoes("IPAdapterCombineEmbeds", "method") if (
    "IPAdapterCombineEmbeds" in OBJECT_INFO) else None
if isinstance(_cmb, list) and "3_REF_FULL_BODY_FACE_OUTFIT" in MODOS_A_EXECUTAR:
    checa(COMBINE_METHOD in _cmb, f"combine '{COMBINE_METHOD}' suportado",
          f"combine '{COMBINE_METHOD}' invalido. Disponiveis: {_cmb}")

print()
print("=" * 64)
print("3. GRAFOS DE CADA MODO")
print("=" * 64)
WORKFLOWS = {}
for modo in MODOS_A_EXECUTAR:
    info = MODOS[modo]
    caminho = pathlib.Path(
        f"/content/ChibiCreate/workflows/{WORKFLOW}/"
        f"{info['workflow_version']}.json")
    raw = caminho.read_bytes()
    wf = {k: v for k, v in json.loads(raw).items() if not k.startswith("_")}
    cls = {k: v["class_type"] for k, v in wf.items()}
    WORKFLOWS[modo] = {"nodes": wf, "sha256": hashlib.sha256(raw).hexdigest(),
                       "version": info["workflow_version"]}
    print(f"\n  [{info['slug']}] {info['workflow_version']}.json "
          f"({len(wf)} nodes)")

    ks = next((k for k, c in cls.items() if c == "KSampler"), None)
    enc = [k for k, c in cls.items() if c == "VAEEncode"]
    loads = [k for k, c in cls.items() if c == "LoadImage"]
    checa(bool(enc) and ks is not None, "VAEEncode e KSampler presentes",
          "grafo incompleto")
    checa("EmptyLatentImage" not in cls.values(),
          "sem EmptyLatentImage (img2img real)",
          "EmptyLatentImage presente: viraria txt2img")
    checa(wf[ks]["inputs"]["latent_image"] == [enc[0], 0],
          "latente vem do VAEEncode", "latente nao vem do VAEEncode")
    fb = wf[enc[0]]["inputs"]["pixels"][0]
    checa(wf[fb]["inputs"]["image"] == "%%REF_FULL_BODY%%",
          "latente parte de full_body", "latente nao parte de full_body")
    checa(cls[wf[ks]["inputs"]["model"][0]] == "IPAdapterEmbeds",
          "MODEL condicionado pelo IP-Adapter",
          "KSampler nao recebe o modelo condicionado")
    # full_body em papel duplo nos DOIS modos
    usos = {cls[k] for k, v in wf.items()
            if any(isinstance(x, list) and x[0] == fb
                   for x in v["inputs"].values())}
    checa({"VAEEncode", "IPAdapterEncoder"} <= usos,
          "full_body em papel duplo (latente + referencia)",
          f"full_body usado so em {usos}")
    checa(len(loads) == info["reference_count"],
          f"{info['reference_count']} imagem(ns) carregada(s)",
          f"esperado {info['reference_count']} LoadImage, achou {len(loads)}")
    placeholders = {wf[k]["inputs"]["image"] for k in loads}
    esperado = {f"%%REF_{r.upper()}%%" for r in info["refs"]}
    checa(placeholders == esperado, "referencias do grafo == declaradas",
          f"grafo usa {placeholders}, esperado {esperado}")
    ausentes = [c for c in set(cls.values()) if c not in OBJECT_INFO]
    checa(not ausentes, "todas as classes existem no servidor",
          f"classes ausentes: {ausentes}")

print()
print("=" * 64)
print("PROMPT")
print("=" * 64)
print("positivo (", PROMPT_SOURCE["positive"], "):", PROMPT[:80])
print("negativo (", PROMPT_SOURCE["negative"], "):", NEGATIVE[:80])

# O prompt-base precisa servir para 100+ personagens. Se descrever a
# waifu_001, a recipe deixa de ser reutilizavel e a identidade passa a vir
# do texto em vez da imagem — que e exatamente o que este lab testa.
# So o POSITIVO e validado: o negativo lista o que evitar, entao termos
# como "horns" ali sao legitimos.
_achados = mr.termos_especificos_no_prompt(PROMPT)
checa(not _achados, "prompt positivo generico (sem termos de personagem)",
      f"prompt positivo contem termos especificos de personagem: {_achados}. "
      "Identidade vem da imagem e das referencias, nao do texto.")
checa(bool(PROMPT.strip()), "prompt positivo nao vazio",
      "prompt positivo vazio")

print()
if falhas:
    raise SystemExit(f"BLOCKED — {len(falhas)} falha(s): {falhas}")
print("Validacao completa. Nenhuma referencia descartada em silencio.")


---

## Celula 9 — executar

Monta o grafo com os parametros e pesos declarados e envia ao ComfyUI.

In [ ]:
#@title 9. Executar o experimento (todos os modos) { display-mode: "form" }
import json, time, copy, hashlib, pathlib, urllib.request, uuid, shutil
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

try:
    VERSAO
except NameError:
    raise SystemExit("BLOCKED — execute a celula 4 (Drive) antes desta.")
if not (VERSAO.get("sha256") and VERSAO.get("sdxl_validated")):
    raise SystemExit("BLOCKED — checkpoint nao validado. Reexecute a celula 4.")
CKPT_PATH = pathlib.Path(VERSAO["comfy_path"])
if not CKPT_PATH.exists():
    raise SystemExit(f"BLOCKED — '{CKPT_PATH}' sumiu. Reexecute a celula 4.")

EXP_DIR = pathlib.Path(
    f"/content/ChibiCreate/experiments/wai_chibi_lab/{EXPERIMENT_DIR_NAME}")
if EXP_DIR.exists():
    raise SystemExit(
        f"BLOCKED — {EXP_DIR} ja existe. Experimentos nunca sao "
        "sobrescritos; reexecute a celula 0 para gerar novo timestamp.")
EXP_DIR.mkdir(parents=True)
(EXP_DIR / "config.json").write_text(json.dumps(CONFIG, indent=2))

COMFY_INPUT = pathlib.Path("/content/ComfyUI/input")
COMFY_INPUT.mkdir(parents=True, exist_ok=True)
nomes = {}
for papel in ("full_body", "face", "outfit"):
    if papel in ENTRADAS:
        nome = f"{CHARACTER_ID}_{papel}.png"
        (COMFY_INPUT / nome).write_bytes(
            pathlib.Path(ENTRADAS[papel]["path"]).read_bytes())
        nomes[papel] = nome
SRC_W, SRC_H = ENTRADAS["full_body"]["size"]

def executar(modo, denoise):
    """Monta, valida, submete e registra UMA execucao."""
    info = MODOS[modo]
    slug = info["slug"]
    sub = f"{slug}_d{denoise:.2f}" if len(DENOISE_VALUES) > 1 else slug
    out_dir = EXP_DIR / sub
    out_dir.mkdir(parents=True)
    (out_dir / "logs").mkdir()

    prefix = f"lab_{CHARACTER_ID}_{sub}_{uuid.uuid4().hex[:8]}"
    subst = {
        "%%CKPT_NAME%%": CKPT_PATH.name,
        "%%PROMPT%%": PROMPT, "%%NEGATIVE_PROMPT%%": NEGATIVE,
        "%%REF_FULL_BODY%%": nomes.get("full_body"),
        "%%REF_FACE%%": nomes.get("face"),
        "%%REF_OUTFIT%%": nomes.get("outfit"),
        "%%WEIGHT_FULL_BODY%%": float(FULL_BODY_WEIGHT),
        "%%WEIGHT_FACE%%": float(FACE_WEIGHT),
        "%%WEIGHT_OUTFIT%%": float(OUTFIT_WEIGHT),
        "%%COMBINE_METHOD%%": COMBINE_METHOD,
        "%%IPADAPTER_WEIGHT%%": float(IPADAPTER_WEIGHT),
        "%%WEIGHT_TYPE%%": WEIGHT_TYPE,
        "%%START_AT%%": float(START_AT), "%%END_AT%%": float(END_AT),
        "%%EMBEDS_SCALING%%": EMBEDS_SCALING,
        "%%IPADAPTER_FILE%%": CFG["ipadapter_models"]["adapter"]["file"],
        "%%CLIP_VISION_FILE%%": CFG["ipadapter_models"]["clip_vision"]["file"],
        "%%SEED%%": int(SEED), "%%STEPS%%": int(STEPS), "%%CFG%%": float(CFG_SCALE),
        "%%SAMPLER%%": SAMPLER, "%%SCHEDULER%%": SCHEDULER,
        "%%DENOISE%%": float(denoise), "%%OUTPUT_PREFIX%%": prefix,
    }
    grafo = copy.deepcopy(WORKFLOWS[modo]["nodes"])
    for node in grafo.values():
        for campo, valor in node["inputs"].items():
            if isinstance(valor, str) and valor in subst:
                v = subst[valor]
                if v is None:
                    raise SystemExit(f"BLOCKED — placeholder {valor} sem valor.")
                node["inputs"][campo] = v
        node.pop("_meta", None)
    resto = [v for n in grafo.values() for v in n["inputs"].values()
             if isinstance(v, str) and v.startswith("%%")]
    assert not resto, f"placeholders nao resolvidos: {resto}"
    (out_dir / "workflow.resolved.json").write_text(json.dumps(grafo, indent=2))

    print(f"\n{'=' * 64}")
    print(f"EXECUTANDO  {sub}   ({info['reference_count']} ref, denoise {denoise})")
    print("=" * 64)
    _enc = next(k for k, v in grafo.items() if v["class_type"] == "VAEEncode")
    _ks = next(k for k, v in grafo.items() if v["class_type"] == "KSampler")
    _ld = grafo[_enc]["inputs"]["pixels"][0]
    print(f"  latente : {grafo[_ld]['inputs']['image']} -> LoadImage[{_ld}]"
          f" -> VAEEncode[{_enc}] -> KSampler[{_ks}]")
    print(f"  model   : IPAdapterEmbeds[{grafo[_ks]['inputs']['model'][0]}]"
          f" (weight {IPADAPTER_WEIGHT}, {WEIGHT_TYPE},"
          f" {START_AT}-{END_AT})")
    print(f"  refs    : {', '.join(info['refs'])}")

    t0 = time.time()
    req = urllib.request.Request(
        "http://127.0.0.1:8188/prompt",
        data=json.dumps({"prompt": grafo}).encode(),
        headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=60) as r:
            pid = json.load(r)["prompt_id"]
    except urllib.error.HTTPError as e:
        detalhe = e.read().decode()[:3000]
        (out_dir / "logs" / "erro.txt").write_text(detalhe)
        print(detalhe)
        raise SystemExit(f"BLOCKED em '{sub}' na submissao — erro acima.")

    while True:
        time.sleep(3)
        with urllib.request.urlopen(
                f"http://127.0.0.1:8188/history/{pid}", timeout=30) as r:
            h = json.load(r)
        if pid in h:
            hist = h[pid]
            break
        if time.time() - t0 > 1800:
            raise SystemExit(f"BLOCKED em '{sub}' — timeout de 30 min.")

    exec_time = round(time.time() - t0, 2)
    if hist.get("status", {}).get("status_str") == "error":
        detalhe = json.dumps(hist["status"], indent=2)[:3000]
        (out_dir / "logs" / "erro.txt").write_text(detalhe)
        print(detalhe)
        raise SystemExit(f"BLOCKED em '{sub}' na execucao — erro acima.")

    saidas = [i for o in hist["outputs"].values() for i in o.get("images", [])]
    assert saidas, f"{sub}: nenhuma imagem produzida"
    origem = (pathlib.Path("/content/ComfyUI/output") /
              (saidas[0].get("subfolder") or "") / saidas[0]["filename"])
    destino = out_dir / "output.png"
    shutil.copy2(origem, destino)
    img = Image.open(destino)

    log_src = pathlib.Path("/content/comfyui_wai.log")
    if log_src.exists():
        shutil.copy2(log_src, out_dir / "logs" / "comfyui.log")
    (out_dir / "logs" / "history.json").write_text(
        json.dumps(hist, indent=2, default=str))

    recipe = {
        "experiment": EXPERIMENT_NAME, "mode": modo, "slug": sub,
        "pipeline": "img2img",
        "primary_image": "full_body", "primary_image_role": "source_image",
        "character": CHARACTER_ID,
        "model": {
            "key": MODEL_KEY, "file": VERSAO["file"],
            "sha256": VERSAO["sha256"], "source": VERSAO["source"],
            "size_bytes": VERSAO["size_bytes"],
            "civitai_model_id": VERSAO["civitai_model_id"],
            "civitai_model_version_id": VERSAO["civitai_model_version_id"],
            "license": CFG["license_name"],
            "commercial_status": CFG["commercial_status"],
            "sdxl_validated": VERSAO["sdxl_validated"],
        },
        "references_declared": info["refs"],
        "references_consumed": info["refs"],
        "reference_count": info["reference_count"],
        "reference_roles": {
            r: (["source_image", "ipadapter_reference"] if r == "full_body"
                else ["ipadapter_reference"]) for r in info["refs"]},
        "dual_role_note": (
            "full_body.png e imagem inicial do img2img (VAEEncode) E "
            "referencia do IP-Adapter (IPAdapterEncoder). Intencional."),
        "references": {
            r: {"file": ENTRADAS[r]["file"],
                "artifact_sha256": ENTRADAS[r]["artifact_sha256"],
                "pixel_sha256": ENTRADAS[r]["pixel_sha256"],
                "size": ENTRADAS[r]["size"],
                "weight": CONFIG["reference_weights"][r]}
            for r in info["refs"]},
        "prompt": PROMPT, "negative_prompt": NEGATIVE,
        "prompt_preset": PROMPT_PRESET, "prompt_type": "generic_chibi_base",
        "character_specific_prompt": False,
        "prompt_source": PROMPT_SOURCE,
        "prompt_manually_edited": PROMPT_EDITADO,
        "parameters": {
            "seed": int(SEED), "steps": int(STEPS), "cfg": float(CFG_SCALE),
            "sampler": SAMPLER, "scheduler": SCHEDULER,
            "denoise": float(denoise),
            "denoise_status": "BASELINE_EXPERIMENTAL",
            "resolution": [SRC_W, SRC_H],
            "resolution_note": "herdada da imagem de partida; sem resize",
            "batch": 1, "vae": "integrado ao checkpoint",
        },
        "ipadapter": {
            "weight": float(IPADAPTER_WEIGHT), "weight_type": WEIGHT_TYPE,
            "start_at": float(START_AT), "end_at": float(END_AT),
            "embeds_scaling": EMBEDS_SCALING,
            "combine_method": (COMBINE_METHOD
                               if info["reference_count"] > 1 else None),
            "assets": (IPADAPTER_META or {}).get("assets"),
            "status": "BASELINE_EXPERIMENTAL",
        },
        "workflow": f"{WORKFLOW}@{WORKFLOWS[modo]['version']}",
        "workflow_sha256": WORKFLOWS[modo]["sha256"],
        "comfyui_commit": COMFY_COMMIT,
        "custom_nodes": ([] if not IPADAPTER_META else
                         [{"repo": IPADAPTER_META["repo"],
                           "commit": IPADAPTER_META["commit"],
                           "revision": IPADAPTER_META["revision"]}]),
        "hardware": json.load(open("/content/gpu_info_wai.json")),
        "execution_time_s": exec_time,
        "artifact_sha256": hashlib.sha256(destino.read_bytes()).hexdigest(),
        "output_pixel_sha256": hashlib.sha256(
            np.array(img.convert("RGBA")).tobytes()).hexdigest(),
        "output_size": list(img.size),
        "determinism_note": (
            "Hash identico entre execucoes NAO e garantido em outra GPU ou "
            "outra versao de torch/ComfyUI."),
    }
    (out_dir / "recipe.json").write_text(json.dumps(recipe, indent=2, default=str))
    (out_dir / "metadata.json").write_text(json.dumps({
        "experiment": EXPERIMENT_NAME, "slug": sub, "mode": modo,
        "denoise": float(denoise), "reference_count": info["reference_count"],
        "execution_time_s": exec_time,
        "artifact_sha256": recipe["artifact_sha256"],
        "output_pixel_sha256": recipe["output_pixel_sha256"],
        "output_size": recipe["output_size"],
    }, indent=2))

    print(f"  -> {exec_time}s | {destino}")
    return {"slug": sub, "mode": modo, "denoise": float(denoise),
            "path": destino, "recipe": recipe}

RESULTADOS = []
for modo in MODOS_A_EXECUTAR:
    for dn in DENOISE_VALUES:
        RESULTADOS.append(executar(modo, dn))

print()
print("=" * 64)
print(f"EXPERIMENTO CONCLUIDO — {len(RESULTADOS)} execucao(oes)")
print("=" * 64)
print("diretorio:", EXP_DIR)
for r in RESULTADOS:
    print(f"  {r['slug']:16} {r['recipe']['execution_time_s']:>7}s  "
          f"{r['recipe']['output_pixel_sha256'][:16]}...")


---

## Celula 10 — recipe e hashes

Registro completo de reprodutibilidade. `artifact_sha256` (bytes do arquivo)
e `output_pixel_sha256` (conteudo dos pixels) sao gravados **separados**.

In [ ]:
#@title 10. Resumo do experimento { display-mode: "form" }
#@markdown Recipe, metadata, workflow resolvido e logs ja foram gravados
#@markdown pela celula 9, um diretorio por execucao. Aqui so conferimos.
import json, pathlib

print("=" * 64)
print("ARTEFATOS GRAVADOS")
print("=" * 64)
print("experimento:", EXPERIMENT_NAME)
print("diretorio  :", EXP_DIR)
print()
for f in sorted(EXP_DIR.rglob("*")):
    if f.is_file():
        print(f"  {f.relative_to(EXP_DIR)}  ({f.stat().st_size:,} bytes)")

print()
print("=" * 64)
print("REPRODUTIBILIDADE")
print("=" * 64)
for r in RESULTADOS:
    rec = r["recipe"]
    print(f"\n[{r['slug']}]")
    print("  checkpoint sha256 :", rec["model"]["sha256"])
    for papel, a in (rec["ipadapter"].get("assets") or {}).items():
        print(f"  {papel:17} : {a['sha256']}  ({a['file']})")
    print("  workflow sha256   :", rec["workflow_sha256"])
    print("  ComfyUI commit    :", rec["comfyui_commit"])
    for cn in rec["custom_nodes"]:
        print(f"  custom node       : {cn['repo']} @ {cn['commit']}")
    print("  seed / denoise    :", rec["parameters"]["seed"], "/",
          rec["parameters"]["denoise"])
    print("  artifact sha256   :", rec["artifact_sha256"])
    print("  pixel sha256      :", rec["output_pixel_sha256"])

# Execucoes com a MESMA configuracao devem dar o mesmo pixel hash.
_por_config = {}
for r in RESULTADOS:
    chave = (r["mode"], r["denoise"])
    _por_config.setdefault(chave, []).append(r)
_repetidas = {k: v for k, v in _por_config.items() if len(v) > 1}
if _repetidas:
    print()
    print("DETERMINISMO (mesma config executada mais de uma vez)")
    for (modo, dn), grupo in _repetidas.items():
        hashes = {g["recipe"]["output_pixel_sha256"] for g in grupo}
        print(f"  {modo} d{dn}: "
              f"{'IDENTICO' if len(hashes) == 1 else 'DIFERENTE'}")
    print("  Escopo: mesma sessao, mesma GPU, mesmas versoes.")


---

## Celula 11 — reprodutibilidade (001 vs 002)

Rode depois de ter executado as Runs 001 e 002.

In [ ]:
#@title 11. Historico de experimentos { display-mode: "form" }
#@markdown Lista todos os experimentos ja rodados, para comparar
#@markdown configuracoes entre si. Nada e sobrescrito.
import json, pathlib

LAB = pathlib.Path("/content/ChibiCreate/experiments/wai_chibi_lab")
if not LAB.exists():
    print("Nenhum experimento ainda.")
else:
    dirs = sorted(d for d in LAB.iterdir() if (d / "config.json").exists())
    print(f"{len(dirs)} experimento(s) em {LAB}")
    print()
    print(f"{'experimento':22} {'denoise':>8} {'ipa':>5} {'wtype':>10} "
          f"{'steps':>5} {'cfg':>5} {'modos'}")
    print("-" * 78)
    for d in dirs:
        c = json.loads((d / "config.json").read_text())
        modos = ",".join(MODOS[m]["slug"] for m in c["modes_to_run"]
                         if m in MODOS)
        print(f"{d.name[-15:]:22} "
              f"{str(c['denoise_values']):>8} "
              f"{c['ipadapter']['weight']:>5} "
              f"{c['ipadapter']['weight_type']:>10} "
              f"{c['sampling']['steps']:>5} {c['sampling']['cfg']:>5} "
              f"{modos}")
    print()
    print("Para comparar dois experimentos, abra os comparison.png de cada.")


---

## Celula 12 — montagem comparativa

Sem ranking automatico. A leitura e humana.

In [ ]:
#@title 12. Comparacao 1 REF x 3 REFS { display-mode: "form" }
import json, pathlib
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

ORIGINAL = Image.open(ENTRADAS["full_body"]["path"]).convert("RGB")

paineis = [(ORIGINAL, "ORIGINAL\n(imagem de partida)")]
for r in RESULTADOS:
    rec = r["recipe"]
    paineis.append((
        Image.open(r["path"]).convert("RGB"),
        f"{r['slug']}\n{rec['reference_count']} ref | denoise "
        f"{rec['parameters']['denoise']}\nipa {rec['ipadapter']['weight']}"
        f" {rec['ipadapter']['weight_type']}"))

n = len(paineis)
fig, ax = plt.subplots(1, n, figsize=(4.3 * n, 6.4))
for a, (im, t) in zip(np.atleast_1d(ax), paineis):
    a.imshow(im); a.set_title(t, fontsize=9); a.axis("off")
plt.suptitle(f"Experimento: {EXPERIMENT_NAME}", fontsize=11)
plt.tight_layout()
COMPARISON = EXP_DIR / "comparison.png"
plt.savefig(COMPARISON, dpi=110, bbox_inches="tight")
plt.show()

# ---- metricas TECNICAS objetivas. Nao sao nota de qualidade. ----
def _metricas(caminho):
    im = Image.open(caminho).convert("RGB")
    arr = np.asarray(im, dtype=np.float32) / 255.0
    cinza = arr.mean(axis=2)
    gy, gx = np.gradient(cinza)
    base = np.asarray(ORIGINAL.resize(im.size), dtype=np.float32) / 255.0
    return {
        "resolution": list(im.size),
        "mean_rgb": [round(float(x), 4) for x in arr.reshape(-1, 3).mean(0)],
        "saturation_mean": round(float(
            (arr.max(2) - arr.min(2)).mean()), 4),
        "edge_density": round(float(np.hypot(gx, gy).mean()), 5),
        "vs_original_abs_diff": round(float(np.abs(arr - base).mean()), 5),
        "vs_original_rmse": round(float(
            np.sqrt(((arr - base) ** 2).mean())), 5),
    }

COMPARISON_JSON = {
    "experiment": EXPERIMENT_NAME,
    "experiment_dir": EXPERIMENT_DIR_NAME,
    "config": CONFIG,
    "note": (
        "Metricas TECNICAS objetivas apenas. Nao sao nota de qualidade e "
        "nao ordenam os resultados: distancia menor da original significa "
        "menos transformacao, nao 'melhor'. Avaliacao artistica e humana."),
    "runs": [],
}
for r in RESULTADOS:
    rec = r["recipe"]
    COMPARISON_JSON["runs"].append({
        "slug": r["slug"], "mode": r["mode"],
        "reference_count": rec["reference_count"],
        "references_consumed": rec["references_consumed"],
        "reference_weights": {k: v["weight"]
                              for k, v in rec["references"].items()},
        "combine_method": rec["ipadapter"]["combine_method"],
        "ipadapter": {k: rec["ipadapter"][k] for k in
                      ("weight", "weight_type", "start_at", "end_at",
                       "embeds_scaling")},
        "parameters": rec["parameters"],
        "workflow": rec["workflow"],
        "workflow_sha256": rec["workflow_sha256"],
        "checkpoint_sha256": rec["model"]["sha256"],
        "ipadapter_assets": {
            k: v.get("sha256") for k, v in
            (rec["ipadapter"].get("assets") or {}).items()},
        "comfyui_commit": rec["comfyui_commit"],
        "custom_nodes": rec["custom_nodes"],
        "execution_time_s": rec["execution_time_s"],
        "artifact_sha256": rec["artifact_sha256"],
        "output_pixel_sha256": rec["output_pixel_sha256"],
        "technical_metrics": _metricas(r["path"]),
    })
(EXP_DIR / "comparison.json").write_text(
    json.dumps(COMPARISON_JSON, indent=2, default=str))

print("=" * 64)
print("METRICAS TECNICAS (nao sao nota de qualidade)")
print("=" * 64)
print(f"{'slug':16} {'refs':>4} {'denoise':>7} {'ipa':>5} "
      f"{'dist.orig':>9} {'edges':>7}")
for run in COMPARISON_JSON["runs"]:
    m = run["technical_metrics"]
    print(f"{run['slug']:16} {run['reference_count']:>4} "
          f"{run['parameters']['denoise']:>7} "
          f"{run['ipadapter']['weight']:>5} "
          f"{m['vs_original_abs_diff']:>9} {m['edge_density']:>7}")

print()
print("  dist.orig: quanto a saida se afastou da imagem de partida.")
print("             MAIOR = mais transformacao. Nao significa melhor.")
print("  edges    : densidade de bordas. Proxy fraco de detalhe/lineart.")
print()
print("=" * 64)
print("[HUMAN REVIEW REQUIRED] — sem ranking automatico")
print("=" * 64)
print("Compare 1 REF x 3 REFS nos eixos:")
for eixo, itens in (
        ("STYLE", ["proporcao chibi atingida", "lineart", "cel shading"]),
        ("IDENTITY", ["rosto", "cabelo", "expressao"]),
        ("DESIGN_PRESERVATION", ["roupa", "capa", "ornamentos",
                                 "acessorios", "cores"])):
    print(f"\n{eixo}")
    for i in itens:
        print("  [ ]", i)
print()
print("salvo:", COMPARISON)
print("salvo:", EXP_DIR / "comparison.json")


In [ ]:
#@title 13. Empacotar o experimento em ZIP { display-mode: "form" }
import json, hashlib, pathlib, shutil, zipfile, datetime

STAGE = pathlib.Path("/content/_wai_zip")
if STAGE.exists():
    shutil.rmtree(STAGE)
shutil.copytree(EXP_DIR, STAGE)

hashes = {}
for f in sorted(STAGE.rglob("*")):
    if f.is_file():
        hashes[str(f.relative_to(STAGE))] = {
            "sha256": hashlib.sha256(f.read_bytes()).hexdigest(),
            "bytes": f.stat().st_size,
        }
(STAGE / "hashes.json").write_text(json.dumps(hashes, indent=2))

L = []
L.append(f"# WAI CHIBI EXPERIMENT LAB — {EXPERIMENT_NAME}\n")
L.append(f"Gerado em {datetime.datetime.now(datetime.timezone.utc).isoformat()}")
L.append(f"Personagem: {CHARACTER_ID}\n")
L.append("> LABORATORIO DE CONFIGURACOES. Nao e pipeline oficial.")
L.append("> O benchmark FLUX nao foi modificado nem reexecutado.\n")

L.append("## Pipeline\n")
L.append("Todas as execucoes sao **img2img real**. O latente inicial vem")
L.append("sempre de `full_body.png`:\n")
L.append("```")
L.append("full_body.png -> LoadImage -> VAEEncode -> KSampler(latent_image)")
L.append("              -> VAEDecode -> output.png")
L.append("```\n")
L.append("O IP-Adapter atua **adicionalmente sobre o MODEL**, nunca")
L.append("substituindo o latente. Nao ha `EmptyLatentImage`.\n")
L.append("`full_body.png` tem **dois papeis** nos dois modos: imagem")
L.append("inicial do img2img e referencia do IP-Adapter.\n")

L.append("## Configuracao\n")
s_ = CONFIG["sampling"]
L.append(f"- seed: `{s_['seed']}` | steps: `{s_['steps']}` | "
         f"cfg: `{s_['cfg']}`")
L.append(f"- sampler/scheduler: `{s_['sampler']}` / `{s_['scheduler']}`")
L.append(f"- denoise: `{CONFIG['denoise_values']}` "
         f"(BASELINE_EXPERIMENTAL, < 1.0 obrigatorio)")
i_ = CONFIG["ipadapter"]
L.append(f"- IP-Adapter: weight `{i_['weight']}`, type "
         f"`{i_['weight_type']}`, start `{i_['start_at']}`, end "
         f"`{i_['end_at']}`, scaling `{i_['embeds_scaling']}`")
L.append(f"- pesos por referencia: {CONFIG['reference_weights']}")
L.append(f"- combine method: `{CONFIG['combine_method']}`\n")

L.append("## Prompt (generico)\n")
L.append(f"- preset: `{CONFIG['prompt_preset']}`")
L.append(f"- `prompt_type`: `{CONFIG['prompt_type']}`")
L.append(f"- `character_specific_prompt`: "
         f"`{CONFIG['character_specific_prompt']}`\n")
L.append(f"> {CONFIG['prompt']}\n")
L.append(f"**Negativo:**\n\n> {CONFIG['negative_prompt']}\n")
L.append("O prompt **nao descreve a personagem**: identidade e design vem")
L.append("da imagem de entrada e das referencias. O preset vale para")
L.append("qualquer personagem, sem edicao manual.\n")

L.append("## Execucoes\n")
L.append("| slug | refs | denoise | ipa weight | tempo | pixel sha256 |")
L.append("|---|---|---|---|---|---|")
for r in RESULTADOS:
    rec = r["recipe"]
    L.append(f"| {r['slug']} | {rec['reference_count']} | "
             f"{rec['parameters']['denoise']} | "
             f"{rec['ipadapter']['weight']} | "
             f"{rec['execution_time_s']}s | "
             f"`{rec['output_pixel_sha256'][:16]}...` |")

L.append("\n## Reprodutibilidade\n")
_um = RESULTADOS[0]["recipe"]
L.append(f"- checkpoint: `{_um['model']['file']}`")
L.append(f"  - sha256: `{_um['model']['sha256']}`")
L.append(f"  - licenca: {_um['model']['license']} — "
         f"{_um['model']['commercial_status']}")
for papel, a in (_um["ipadapter"].get("assets") or {}).items():
    L.append(f"- {papel}: `{a['file']}` — sha256 `{a['sha256']}` "
             f"({a['license']})")
L.append(f"- ComfyUI: `{_um['comfyui_commit']}`")
for cn in _um["custom_nodes"]:
    L.append(f"- custom node: {cn['repo']} @ `{cn['commit']}` "
             f"({cn['revision']})")
hw = _um["hardware"]
L.append(f"- GPU: {hw.get('name')} | VRAM {hw.get('vram_total_gb')} GB | "
         f"RAM {hw.get('ram_gb')} GB")
L.append(f"- CUDA {hw.get('cuda')} | torch {hw.get('torch')} | "
         f"python {hw.get('python')}\n")
L.append("Hash identico entre execucoes nao e garantido em outra GPU ou")
L.append("outra versao de torch/ComfyUI. Nao afirmamos determinismo")
L.append("absoluto.\n")

L.append("## Leitura\n")
L.append("`comparison.json` traz metricas **tecnicas** (distancia da")
L.append("original, densidade de bordas). Elas **nao sao nota de")
L.append("qualidade** e nao ordenam os resultados: distancia maior")
L.append("significa mais transformacao, nao 'melhor'.\n")
L.append("[HUMAN REVIEW REQUIRED] Comparar 1 REF x 3 REFS nos eixos")
L.append("STYLE / IDENTITY / DESIGN_PRESERVATION e decisao humana.")
(STAGE / "RELATORIO.md").write_text("\n".join(L))

ZIP_PATH = pathlib.Path(
    f"/content/wai_chibi_lab_{EXPERIMENT_DIR_NAME}.zip")
if ZIP_PATH.exists():
    ZIP_PATH.unlink()
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(STAGE.rglob("*")):
        if f.is_file():
            z.write(f, f.relative_to(STAGE))

print("=" * 64)
print("ZIP PRONTO")
print("=" * 64)
print("caminho :", ZIP_PATH)
print("tamanho :", f"{ZIP_PATH.stat().st_size:,} bytes")
print()
with zipfile.ZipFile(ZIP_PATH) as z:
    for n in sorted(z.namelist()):
        print("  ", n)

try:
    from google.colab import files
    files.download(str(ZIP_PATH))
    print()
    print("Download iniciado. Se o navegador bloquear, use o painel de")
    print("arquivos a esquerda:", ZIP_PATH)
except Exception as exc:
    print()
    print(f"[aviso] download automatico indisponivel ({type(exc).__name__}).")
    print("Baixe pelo painel de arquivos a esquerda:", ZIP_PATH)


---

## PARE AQUI

Concluido: Runs 001, 002, 003, receitas, hashes e o ZIP.

O objetivo desta rodada era **separar tres causas**, nao escolher vencedor:

| observacao | causa | proximo passo |
|---|---|---|
| Run 001 ja suja | **(A)** configuracao / checkpoint | parar e reportar — IP-Adapter nao participou |
| 001 limpa, 003 suja | **(B)** WAI + IP-Adapter | investigar **um** fator por vez |
| ambas limpas | **(C)** questao artistica | comparar com o FLUX |

### [HUMAN REVIEW REQUIRED]

- A imagem esta tecnicamente limpa?
- Concluir com **uma** marca: `PROMISING` / `INSUFFICIENT` / `BLOCKED`.

**Nao** fazer agora: calibrar pesos, Hires fix, outros checkpoints, Pony,
outro IP-Adapter, inpainting, design transfer, ou qualquer mudanca no FLUX.
